# Service Health Monitor

Reads the snapshot artifacts `snapshot.py` already computed, `flags.csv` and `events.json`, and the raw panel in `data/service_metrics.csv`. This notebook never re-runs a detector; that's the whole point of the snapshot architecture (see the project README). If a detector's logic changes, `snapshot.py` reruns and this notebook just reads the new files, same as it reads these.

Everything here is synthetic data generated in-repo. No proprietary data, metrics, or results from any employer are used or implied.

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import sys
sys.path.insert(0, str(Path.cwd().parent / "src"))
from style import set_style, style_ax, savefig, SLATE, MUTED_TEAL, MUTED_AMBER, MUTED_RED, GREY

BASE = Path.cwd().parent
FIG_DIR = BASE / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
set_style()

df = pd.read_csv(BASE / "data" / "service_metrics.csv")
flags = pd.read_csv(BASE / "snapshot" / "flags.csv")
events = json.loads((BASE / "snapshot" / "events.json").read_text())

METRICS = ["p95_latency_ms", "error_rate", "request_volume", "queue_depth"]
print(f"{len(df)} days, {len(flags)} flag rows, {len(events)} alert event(s)")

120 days, 1920 flag rows, 7 alert event(s)


## 1. The raw metrics

Four daily metrics for a fictional web service: p95 latency, error rate, request volume, queue depth. Somewhere in here are four different kinds of trouble, each shaped so a different detector is the one that actually catches it.

In [2]:
LABELS = {
    "p95_latency_ms": "p95 latency (ms)",
    "error_rate": "error rate",
    "request_volume": "request volume",
    "queue_depth": "queue depth",
}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, metric in zip(axes.flat, METRICS):
    ax.plot(df["day"], df[metric], color=SLATE, linewidth=1.2)
    style_ax(ax, title=LABELS[metric], xlabel="Day", ylabel=None, grid_axis="y")
fig.suptitle("Figure 1. Four metrics, four different kinds of trouble", fontsize=15, fontfamily="Lora", color="#ECECEA", x=0.02, ha="left")
savefig(fig, FIG_DIR / "raw_metrics.png", footnote="Source: synthetic service-health panel, 120 days")

## 2. What the detector engine caught

Every metric runs against all four detector types (threshold, z-score, trend-break, data-gap), see `src/detectors.py`. The markers below are `flags.csv`, exactly what the pipeline wrote, not a live recomputation.

In [3]:
DETECTOR_COLORS = {"threshold": MUTED_RED, "zscore": MUTED_AMBER, "trend_break": MUTED_TEAL, "data_gap": GREY}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, metric in zip(axes.flat, METRICS):
    ax.plot(df["day"], df[metric], color=SLATE, linewidth=1.0, zorder=2, alpha=0.8)
    metric_flags = flags[(flags["metric"] == metric) & (flags["flagged"])]
    for detector, group in metric_flags.groupby("detector"):
        y = df.set_index("day").loc[group["day"], metric]
        ax.scatter(group["day"], y, color=DETECTOR_COLORS[detector], s=26, zorder=3, label=detector)
    style_ax(ax, title=LABELS[metric], xlabel="Day", ylabel=None, grid_axis="y")
    ax.legend(fontsize=8, loc="upper left")
fig.suptitle("Figure 2. Flagged days by detector type", fontsize=15, fontfamily="Lora", color="#ECECEA", x=0.02, ha="left")
savefig(fig, FIG_DIR / "detector_flags.png", footnote="Source: snapshot/flags.csv, written by src/snapshot.py")

Two things worth noticing. `request_volume`'s decline only gets flagged around day 105, roughly two weeks after it started on day 95, because `trend_break` needs enough low days inside its 7-day window before the comparison to the 30-day baseline crosses 15%. And `p95_latency_ms` gets flagged by `trend_break` again on days 55, 57, and 76, not new incidents, the original day-45 spike is still sitting inside the 30-day baseline window `trend_break` compares against, which temporarily makes ordinary post-spike readings look like a drop by comparison. That's a real characteristic of window-based baselines, a resolved anomaly can echo until it fully ages out of the comparison window, and it's exactly why the engine runs `threshold` and `zscore` alongside `trend_break` instead of relying on one method: they don't share this blind spot, since each one compares a day only to its own recent history, not to a window that can still contain the old anomaly.

## 3. From flags to alerts: dedup in action

`error_rate` is flagged on 45 of the 120 days, every day from day 75 onward, once the step change hits and never recovers. A naive alert-per-flagged-day design would have sent 45 alerts. `find_alert_events()` collapses that into one, on the day the run starts; see `src/alerts.py`.

In [4]:
flagged_days = flags[flags["flagged"]].groupby("metric")["day"].nunique()
alert_counts = pd.Series({m: sum(1 for e in events if e["metric"] == m) for m in METRICS})

comparison = pd.DataFrame({"flagged days": flagged_days, "alert events": alert_counts}).fillna(0).astype(int)
print(comparison)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = range(len(METRICS))
width = 0.35
ax.bar([i - width / 2 for i in x], [comparison.loc[m, "flagged days"] for m in METRICS], width, color=SLATE, label="Flagged days")
ax.bar([i + width / 2 for i in x], [comparison.loc[m, "alert events"] for m in METRICS], width, color=MUTED_RED, label="Alert events (deduped)")
ax.set_xticks(list(x))
ax.set_xticklabels([LABELS[m] for m in METRICS], fontsize=9.5)
style_ax(ax, title="Figure 3. Dedup turns 45 flagged days into 1 alert", ylabel="Count")
ax.legend(fontsize=9)
savefig(fig, FIG_DIR / "alert_dedup.png", footnote="Source: snapshot/flags.csv vs. snapshot/events.json")

                flagged days  alert events
error_rate                45             1
p95_latency_ms            35             4
queue_depth                3             1
request_volume            15             1


## 4. The alert log

Real output from `src/snapshot.py`, `ConsoleChannel` and `FileChannel` both received every event below (see `snapshot/alert_log.jsonl` for the file channel's copy).

In [5]:
for event in events:
    print(f"day {event['day']:>3}  {event['metric']:<16} flagged by {', '.join(event['detectors'])}")

day  45  p95_latency_ms   flagged by threshold, trend_break, zscore
day  55  p95_latency_ms   flagged by trend_break
day  55  queue_depth      flagged by data_gap
day  57  p95_latency_ms   flagged by trend_break
day  75  error_rate       flagged by threshold, trend_break, zscore
day  76  p95_latency_ms   flagged by trend_break
day 105  request_volume   flagged by trend_break


## Summary

Four anomaly shapes, four detectors, one engine: a hard SLA breach (`threshold`), a statistical outlier against a rolling baseline (`zscore`), a gradual decline that needs a longer comparison window to surface (`trend_break`), and a reporting gap that isn't a bad value at all (`data_gap`). Running all four against every metric caught all four injected anomalies, at the cost of a few explainable echo detections from `trend_break`'s baseline window. Alert dedup turned 45 consecutive flagged days into a single alert. None of this depended on a live data source or a running dashboard process, `snapshot.py` computed it once and everything downstream, this notebook included, just reads what it wrote.